# EMERALD — Traffic Violation Detection System

An automated pipeline for detecting and logging traffic violations from video footage using YOLOv8 for object detection, ByteTrack for multi-object tracking, and EasyOCR for license plate recognition.

## Pipeline Architecture

The system is organized as a sequential six-stage pipeline:

1. **Frame Ingestion** — video decoding and temporal subsampling (`FrameIngestion`)
2. **Preprocessing** — optional CLAHE contrast enhancement and unsharp masking (`preprocess_frame`)
3. **Detection** — YOLOv8 inference with per-class confidence floors and shape-based false-positive filters (`Detector`)
4. **Tracking** — dual ByteTrack instances with class-group-specific parameters (`Tracker`)
5. **Violation Reasoning** — geometric rules over calibrated lane polygons and track state (`ViolationEngine`)
6. **Evidence Storage** — annotated frame export and JSONL event log (`EvidenceStore`)

A FastAPI server exposes the pipeline over HTTP, accessible from a local HTML control panel via an ngrok tunnel.

## Violation Types

| Code | Description | Applicable Classes |
|------|-------------|--------------------|
| `no_helmet` | Rider detected without helmet across vote window | Bicycle, Motorcycle |
| `triple_riding` | More than two riders on a single two-wheeler | Bicycle, Motorcycle |
| `red_light` | Vehicle crosses stop-line polygon while light is red | All vehicles |
| `illegal_parking` | Vehicle stationary in a no-parking zone for ≥ threshold | All vehicles |
| `wrong_side` | Vehicle motion direction opposes the assigned lane direction | All vehicles |

## Changelog

**v8.1** — Introduced dual `ByteTrack` instances to resolve the conflict between permissive tracking parameters required for motorcycles (detection floor 0.18 conf, single-frame confirmation) and strict parameters required for large vehicles (activation threshold 0.40, two-frame confirmation). Track IDs are partitioned by an offset of 50,000 to prevent collision between the two independent counters.

**v8** — Added minimum bounding-box area and aspect-ratio filters on the car/bus/truck classes to reject road-paint markings falsely detected as vehicles. Introduced `TRACK_MAX_AGE_FRAMES` pruning in `Tracker.update()` and `TRACK_DISPLAY_AGE` gating in `draw_dashboard_overlay` to eliminate ghost track overlays after a vehicle exits frame.

**v3 (golden zone patch)** — Corrected a confidence-floor regression that set car/bus/truck floors above the baseline threshold, reducing large-vehicle recall. Fixed a fallback merge bug where the two-wheeler fallback detector could silently discard car/bus/truck detections.

## 1. Environment Verification

In [1]:
import torch

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")


CUDA Available: True
GPU: Tesla T4
VRAM: 15.6 GB


## 2. Dependency Installation

Install `ultralytics` (YOLOv8), `supervision` (ByteTrack, detection utilities), and `easyocr` (license plate OCR).

In [2]:
!pip install ultralytics supervision easyocr --quiet
import supervision as sv
print(f"Supervision version: {sv.__version__}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.6/273.6 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 12.0 MB/s eta 0:00:00
Supervision version: 0.29.0.post0


## 3. Configuration

All tunable parameters are consolidated here. Adjust thresholds, module toggles, and tracking settings before running the pipeline.

In [3]:
import cv2
import numpy as np
import json, os, time, re
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from collections import defaultdict, deque

import supervision as sv
from ultralytics import YOLO
import easyocr

print("All imports OK")

# Output directory for annotated frames and the violation log
OUTPUT_DIR = Path("violation_output")
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / "frames").mkdir(exist_ok=True)

# ── Sampling ──────────────────────────────────────────────────────────────────
FRAME_SAMPLE_RATE       = 2        # process every Nth frame
ADAPTIVE_SKIP           = True     # enable dynamic rate adjustment under GPU load

MAIN_MODEL_NAME         = "yolov8n.pt"

# ── Detection thresholds ──────────────────────────────────────────────────────
# Global floor passed to YOLO; per-class floors below override it post-inference.
YOLO_CONF_THRESHOLD     = 0.25
YOLO_IMGSZ              = 640

# Per-class confidence floors applied after YOLO inference.
# Two-wheelers use a lower floor to improve recall at distance/occlusion.
# Large-vehicle floors were raised to suppress road-paint false positives.
CLASS_CONF_FLOORS       = {
    0:  0.30,   # person
    1:  0.18,   # bicycle
    2:  0.35,   # car       (raised 0.25 → 0.35; road-paint FP suppression)
    3:  0.18,   # motorcycle
    5:  0.30,   # bus       (raised 0.25 → 0.30; spurious blob suppression)
    7:  0.30,   # truck     (raised 0.25 → 0.30; spurious blob suppression)
    9:  0.28,   # traffic light
}

MOTO_MIN_BOX_AREA_FRAC  = 0.0004   # reject motorcycle detections smaller than this fraction of frame area

# ── Large-vehicle shape filters ───────────────────────────────────────────────
# Road paint markings are distinguishable from real vehicles by their small
# bounding-box area and extreme width-to-height aspect ratios.
CAR_MIN_BOX_AREA_FRAC  = 0.0006   # minimum box area as fraction of frame (~51×51 px at 1920 px wide)
CAR_ASPECT_RATIO_MIN   = 0.25     # minimum w/h (rejects abnormally tall detections)
CAR_ASPECT_RATIO_MAX   = 5.00     # maximum w/h (rejects flat horizontal markings)

# ── Track staleness budgets ───────────────────────────────────────────────────
TRACK_MAX_AGE_FRAMES   = 30       # frames before a TrackState entry is purged
TRACK_DISPLAY_AGE      = 15       # frames before an on-screen bounding box is hidden

# ── ByteTrack parameters: two-wheelers and persons ───────────────────────────
# Permissive settings accommodate low confidence scores and inter-frame
# occlusion common with motorcycles.
MOTO_TRACK_ACTIVATION_THRESH = 0.25   # minimum detection confidence to initialise a track
MOTO_LOST_TRACK_BUFFER       = 20     # frames a lost track is retained before deletion
MOTO_MIN_CONSECUTIVE_FRAMES  = 1      # single-frame confirmation threshold

# ── ByteTrack parameters: large vehicles ─────────────────────────────────────
# Strict settings prevent road-paint ghost detections from spawning tracks,
# since real vehicles produce consistently high-confidence, multi-frame detections.
LV_TRACK_ACTIVATION_THRESH   = 0.40
LV_LOST_TRACK_BUFFER         = 15
LV_MIN_CONSECUTIVE_FRAMES    = 2

# Large-vehicle track IDs are offset to prevent collision with moto/person IDs,
# as each ByteTrack instance maintains an independent counter starting at 1.
LV_TRACK_ID_OFFSET           = 50_000

# ── Helmet detection ──────────────────────────────────────────────────────────
HELMET_CONF             = 0.20   # per-frame confidence floor for helmet model
HELMET_VOTE_WINDOW      = 5      # number of recent frames used for voting
HELMET_VOTE_THRESH      = 0.40   # fraction of frames that must show helmet for positive decision
HELMET_PAD_TOP_FRAC     = 0.90   # upward crop extension as fraction of vehicle box height
HELMET_PAD_SIDE_FRAC    = 0.20   # lateral crop extension as fraction of vehicle box width

# ── Violation confirmation ────────────────────────────────────────────────────
MIN_CONFIRM_FRAMES      = 3      # minimum track history before any violation is evaluated
PARKING_STILL_SECS      = 5      # stationary duration threshold for illegal parking
PIXEL_MOVE_THRESH       = 8      # pixel displacement below which a vehicle is considered stationary

WRONG_SIDE_VOTE_WINDOW  = 5      # frames of direction history used for wrong-side voting
WRONG_SIDE_VOTE_THRESH  = 0.6    # fraction of frames agreeing on wrong direction to trigger flag

VIDEO_SOURCE            = 0      # default source index (webcam); overridden at runtime

# ── Debugging ─────────────────────────────────────────────────────────────────
DEBUG_DETECTOR          = False  # print per-class detection counts when True
DEBUG_EVERY_N_FRAMES    = 10

print(f"Output  -> {OUTPUT_DIR.resolve()}")
print(f"Main model: {MAIN_MODEL_NAME}  |  imgsz={YOLO_IMGSZ}")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
All imports OK
Output  -> /content/violation_output
Main model: yolov8n.pt  |  imgsz=640


In [4]:
# ── Violation module toggles ──────────────────────────────────────────────────
# Set any flag to False to disable that detection module globally.
ENABLE_NO_PARKING    = True
ENABLE_STOP_LINE     = True
ENABLE_HELMET        = True
ENABLE_WRONG_SIDE    = True
ENABLE_TRIPLE_RIDING = True

# When True, per-frame CLAHE and unsharp masking are skipped.
# YOLOv8's internal batch normalisation provides sufficient contrast handling
# for live feeds; full preprocessing is reserved for offline recorded video.
LIVE_FEED_MODE       = True


## 4. Data Source

Mount Google Drive and verify the input video path.

In [5]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [6]:
import os

VIDEO_PATH = "VIDEO PATH"
print("File exists:", os.path.exists(VIDEO_PATH))


File exists: True


## 5. Scene Calibration

`SceneCalibration` stores the polygonal regions that define lane boundaries, no-parking zones, and stop lines for a given camera view. All violation rules operate on this geometry.

`make_demo_calibration()` returns a hardcoded calibration for the reference 2560×1440 footage. For a different camera angle, replace the polygon vertex coordinates accordingly.

`point_in_polygon()` wraps `cv2.pointPolygonTest` and is used throughout the violation engine to test whether a track's leading edge falls within a region of interest.

In [7]:
@dataclass
class SceneCalibration:
    lane_A:             np.ndarray
    lane_B:             np.ndarray
    no_parking_zones:   List[np.ndarray]
    lane_A_direction:   Tuple[float, float]
    lane_B_direction:   Tuple[float, float]
    frame_width:        int = 1280
    frame_height:       int = 720
    lane_A_stop_line:   Optional[np.ndarray] = None
    lane_B_stop_line:   Optional[np.ndarray] = None


def make_demo_calibration(frame_w: int, frame_h: int) -> SceneCalibration:
    lane_A = np.array([
        [193,1439],[1004,1],[1230,1],[2448,1440],[795,1440]
    ], dtype=np.int32)

    lane_B = np.array([
        [967,0],[78,1439],[0,1439],[0,1435],
        [2,409],[777,1],[834,0]
    ], dtype=np.int32)

    parking_zone = np.array([
        [1319,34],[1259,30],[2452,1440],[2553,1440],
        [2556,1129],[2393,947],[2016,611]
    ], dtype=np.int32)

    lane_A_stop_line = np.array([
        [int(frame_w*0.1), int(frame_h*0.8)],
        [int(frame_w*0.4), int(frame_h*0.8)],
        [int(frame_w*0.4), int(frame_h*0.85)],
        [int(frame_w*0.1), int(frame_h*0.85)],
    ], dtype=np.int32)

    lane_B_stop_line = np.array([
        [int(frame_w*0.6), int(frame_h*0.8)],
        [int(frame_w*0.9), int(frame_h*0.8)],
        [int(frame_w*0.9), int(frame_h*0.85)],
        [int(frame_w*0.6), int(frame_h*0.85)],
    ], dtype=np.int32)

    return SceneCalibration(
        lane_A=lane_A, lane_B=lane_B,
        no_parking_zones=[parking_zone],
        lane_A_direction=(0.0, -1.0),
        lane_B_direction=(0.0, 1.0),
        lane_A_stop_line=lane_A_stop_line,
        lane_B_stop_line=lane_B_stop_line,
        frame_width=frame_w,
        frame_height=frame_h,
    )


def point_in_polygon(point: Tuple[int,int], polygon: np.ndarray) -> bool:
    return cv2.pointPolygonTest(polygon, (float(point[0]), float(point[1])), False) >= 0


print("SceneCalibration ready")


SceneCalibration ready


In [8]:
calib = make_demo_calibration(2560, 1440)
print("lane_A shape:", calib.lane_A.shape)
print("lane_B shape:", calib.lane_B.shape)
print("Directions  :", calib.lane_A_direction, calib.lane_B_direction)


lane_A shape: (5, 2)
lane_B shape: (7, 2)
Directions  : (0.0, -1.0) (0.0, 1.0)


## 6. Frame Ingestion and Preprocessing

`FrameIngestion` wraps `cv2.VideoCapture`, applies temporal subsampling at `FRAME_SAMPLE_RATE`, and yields `(frame_index, timestamp_sec, raw_frame)` tuples.

`preprocess_frame` operates in two modes controlled by `LIVE_FEED_MODE`. In offline mode (`lite=False`), it applies CLAHE in LAB colour space followed by an unsharp mask to improve detector performance on low-contrast or shadowed regions. In live mode (`lite=True`), the frame is returned unmodified to reduce per-frame latency.

In [9]:
def preprocess_frame(frame: np.ndarray, lite: bool = False) -> np.ndarray:
    """
    Two modes:
    • lite=True  (live feed) — skip CLAHE/sharpen; just return the frame as-is.
      YOLOv8s internal batch-norm handles contrast well enough on its own,
      and skipping this saves ~3 ms per frame.
    • lite=False (offline)   — full CLAHE + unsharp mask for best visual quality.
    """
    if lite:
        return frame

    lab   = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_eq  = clahe.apply(l)
    enhanced = cv2.cvtColor(cv2.merge([l_eq, a, b]), cv2.COLOR_LAB2BGR)
    blurred  = cv2.GaussianBlur(enhanced, (0, 0), 3)
    return cv2.addWeighted(enhanced, 1.5, blurred, -0.5, 0)


class FrameIngestion:
    """Stage 1: Decode video, emit sampled timestamped frames."""

    def __init__(self, source, sample_rate: int = FRAME_SAMPLE_RATE):
        self.cap         = cv2.VideoCapture(source)
        self.sample_rate = sample_rate
        self.fps         = self.cap.get(cv2.CAP_PROP_FPS) or 25.0
        self.frame_idx   = 0

    @property
    def resolution(self) -> Tuple[int, int]:
        w = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        return w, h

    def frames(self):
        """Generator: yields (frame_index, timestamp_sec, raw_frame)."""
        while self.cap.isOpened():
            ret, frame = self.cap.read()
            if not ret:
                break
            self.frame_idx += 1
            if self.frame_idx % self.sample_rate != 0:
                continue
            yield self.frame_idx, self.frame_idx / self.fps, frame

    def release(self):
        self.cap.release()


print("Ingestion + preprocessing ready")


Ingestion + preprocessing ready


## 7. Object Detection

`Detector` wraps YOLOv8 and applies a two-stage filtering pass after inference. First, per-class confidence floors from `CLASS_CONF_FLOORS` are applied. Second, geometric filters reject motorcycle detections that are too small and large-vehicle detections that match the area or aspect-ratio signature of road-paint markings.

Additional methods handle traffic light state classification via HSV colour masking and license plate region cropping for downstream OCR.

In [10]:
COCO_VEHICLE_IDS   = {1: "bicycle", 2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}
COCO_PERSON_ID     = 0
COCO_TRAFFIC_LIGHT = 9
TWO_WHEELER_IDS    = {1, 3}
LARGE_VEHICLE_IDS  = {2, 5, 7}

def _box_area_frac(box: np.ndarray, frame_shape) -> float:
    x1, y1, x2, y2 = box
    return ((x2 - x1) * (y2 - y1)) / (frame_shape[0] * frame_shape[1])

class Detector:
    def __init__(self, model_name: str = MAIN_MODEL_NAME):
        print(f"Loading main model: {model_name} ...")
        self.model = YOLO(model_name)
        self._frame_h = None
        self._frame_w = None
        self._frame_count = 0

    def detect(self, frame: np.ndarray) -> sv.Detections:
        h, w = frame.shape[:2]
        self._frame_h, self._frame_w = h, w
        self._frame_count += 1
        do_debug = (DEBUG_DETECTOR and self._frame_count % DEBUG_EVERY_N_FRAMES == 0)

        results = self.model(frame, conf=0.10, imgsz=YOLO_IMGSZ, verbose=False)[0]
        detections = sv.Detections.from_ultralytics(results)

        if do_debug and detections.class_id is not None:
            for cid, conf in zip(detections.class_id, detections.confidence):
                if int(cid) in LARGE_VEHICLE_IDS:
                    print(f"[DEBUG frame {self._frame_count}] Raw {COCO_VEHICLE_IDS[int(cid)]} found! conf={conf:.3f}")

        if detections.class_id is not None and len(detections) > 0:
            keep_mask = np.ones(len(detections), dtype=bool)
            for i, (cid, conf, box) in enumerate(zip(detections.class_id, detections.confidence, detections.xyxy)):
                floor = CLASS_CONF_FLOORS.get(int(cid), YOLO_CONF_THRESHOLD)
                if conf < floor:
                    keep_mask[i] = False
                    continue
                # Reject motorcycle detections whose bounding box is too small to be a real vehicle
                if int(cid) == 3 and _box_area_frac(box, (h, w)) < MOTO_MIN_BOX_AREA_FRAC:
                    keep_mask[i] = False
                    continue
                # Reject large-vehicle detections that match the profile of road paint markings:
                # real vehicles have non-trivial area and aspect ratios within normal bounds.
                if int(cid) in LARGE_VEHICLE_IDS:
                    bw = box[2] - box[0]
                    bh = box[3] - box[1]
                    if _box_area_frac(box, (h, w)) < CAR_MIN_BOX_AREA_FRAC:
                        keep_mask[i] = False
                        continue
                    if bh > 0:
                        aspect = bw / bh
                        if not (CAR_ASPECT_RATIO_MIN <= aspect <= CAR_ASPECT_RATIO_MAX):
                            keep_mask[i] = False
                            continue
            detections = detections[keep_mask]

        return detections

    def traffic_light_color(self, frame: np.ndarray, box: np.ndarray) -> str:
        x1, y1, x2, y2 = map(int, box)
        crop = frame[y1:y2, x1:x2]
        if crop.size == 0: return "unknown"
        hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
        masks = {
            "red": cv2.inRange(hsv,(0,120,120),(10,255,255)) | cv2.inRange(hsv,(160,120,120),(180,255,255)),
            "yellow": cv2.inRange(hsv,(15,120,120),(35,255,255)),
            "green":  cv2.inRange(hsv,(40,80,80),(90,255,255)),
        }
        scores = {c: m.sum() for c, m in masks.items()}
        best   = max(scores, key=scores.get)
        return best if scores[best] > 500 else "unknown"

    def crop_plate_region(self, frame: np.ndarray, box: np.ndarray) -> Optional[np.ndarray]:
        x1, y1, x2, y2 = map(int, box)
        h = y2 - y1
        plate_y1 = y1 + int(h * 0.75)
        cx = (x1 + x2) // 2
        half_w = max(int((x2 - x1) * 0.35), 30)
        crop = frame[plate_y1:y2, max(cx-half_w,0):min(cx+half_w, frame.shape[1])]
        return crop if crop.size > 0 else None

detector = Detector(MAIN_MODEL_NAME)


Loading main model: yolov8n.pt ...


## 8. Helmet Detection Model

A dedicated two-class YOLOv8 model classifies the rider-head crop as helmeted or unhelmeted. The model is loaded from the `Juliowiwiwiwi/Bike-Helmet-Detction-Model` repository. Per-frame predictions feed into a sliding-window vote stored in `TrackState.helmet_seen`; a violation is raised only when the vote fraction falls below `HELMET_VOTE_THRESH` over at least half of the `HELMET_VOTE_WINDOW`.

In [11]:
# Clone the custom two-class helmet detection model (helmet / no helmet).
# The confidence floor is set low (HELMET_CONF = 0.20) because false positives
# are absorbed by multi-frame voting in TrackState.helmet_decision().
!git clone --depth 1 https://github.com/Juliowiwiwiwi/Bike-Helmet-Detction-Model.git helmet_src -q

helmet_model = YOLO("helmet_src/Weights/best.pt")
print("Helmet model classes:", helmet_model.names)

# Map class indices to a set for O(1) membership tests at inference time
HELMET_PRESENT_CLASSES = {
    idx for idx, name in helmet_model.names.items()
    if "Without" not in name and "without" not in name
}
print("Helmet-present class ids:", HELMET_PRESENT_CLASSES)


Helmet model classes: {0: 'With Helmet', 1: 'Without Helmet'}
Helmet-present class ids: {0}


## 9. Rider Region Extraction

`crop_rider_region` computes the spatial crop submitted to the helmet classifier. The crop extends above the top of the YOLO bounding box to account for riders whose helmets protrude beyond the detected vehicle boundary, and is padded laterally to include both sides of the head.

In [12]:
def crop_rider_region(frame: np.ndarray, vehicle_box: np.ndarray) -> Optional[np.ndarray]:
    """
    Crop the rider-head region from a two-wheeler bounding box.

    YOLOv8's motorcycle box encloses the entire vehicle from wheel to handlebar.
    The rider's helmet occupies the upper 30–50% of that box and may extend
    above the box boundary when the motorcycle is close to the camera.

    The crop is extended upward by HELMET_PAD_TOP_FRAC * box_height and
    laterally by HELMET_PAD_SIDE_FRAC * box_width, then clamped to frame
    boundaries. Only the upper 60% of the extended region is retained to
    exclude the engine and wheel area from the helmet classifier input.
    """
    x1, y1, x2, y2 = map(int, vehicle_box)
    h = y2 - y1
    w = x2 - x1

    pad_top  = int(h * HELMET_PAD_TOP_FRAC)
    pad_side = int(w * HELMET_PAD_SIDE_FRAC)

    crop_y1 = max(0,              y1 - pad_top)
    crop_y2 = min(frame.shape[0], y1 + int(h * 0.60))
    crop_x1 = max(0,              x1 - pad_side)
    crop_x2 = min(frame.shape[1], x2 + pad_side)

    crop = frame[crop_y1:crop_y2, crop_x1:crop_x2]
    return crop if crop.size > 0 else None


## 10. Tracking

`TrackState` accumulates bounding boxes, confidences, timestamps, and per-frame binary signals (helmet seen, direction) for each active track. Decision methods apply sliding-window voting to reduce noise before raising violations.

`Tracker` partitions incoming detections by class group and routes them to the appropriate ByteTrack instance. The resulting tracks are merged into a single `states` dictionary with collision-free IDs.

In [13]:
@dataclass
class TrackState:
    """Accumulated per-track state across frames."""
    track_id:       int
    class_id:       int
    class_name:     str
    boxes:          List[np.ndarray]  = field(default_factory=list)
    confidences:    List[float]       = field(default_factory=list)
    frame_indices:  List[int]         = field(default_factory=list)
    timestamps:     List[float]       = field(default_factory=list)
    rider_counts:   List[int]         = field(default_factory=list)

    # Bounded deques prevent unbounded memory growth over long tracks
    helmet_seen:    deque = field(
        default_factory=lambda: deque(maxlen=HELMET_VOTE_WINDOW)
    )
    direction_votes: deque = field(
        default_factory=lambda: deque(maxlen=WRONG_SIDE_VOTE_WINDOW)
    )

    def latest_box(self) -> Optional[np.ndarray]:
        return self.boxes[-1] if self.boxes else None

    def center(self, box: np.ndarray) -> Tuple[float, float]:
        x1, y1, x2, y2 = box
        return ((x1+x2)/2, (y1+y2)/2)

    def displacement(self) -> float:
        if len(self.boxes) < 2: return 0.0
        c1 = self.center(self.boxes[-2])
        c2 = self.center(self.boxes[-1])
        return float(np.linalg.norm(np.array(c2) - np.array(c1)))

    def velocity_vector(self) -> Tuple[float, float]:
        if len(self.boxes) < 2: return (0.0, 0.0)
        c1 = self.center(self.boxes[-2])
        c2 = self.center(self.boxes[-1])
        return (c2[0]-c1[0], c2[1]-c1[1])

    def helmet_decision(self) -> Optional[bool]:
        """
        Multi-frame voting over the last HELMET_VOTE_WINDOW frames.

        Returns True if the helmet-present fraction >= HELMET_VOTE_THRESH,
        False if below threshold, or None if insufficient history exists.
        Requires at least half the vote window to be populated before committing.
        """
        votes = list(self.helmet_seen)
        if len(votes) < max(2, HELMET_VOTE_WINDOW // 2):
            return None
        helmet_fraction = sum(votes) / len(votes)
        return helmet_fraction >= HELMET_VOTE_THRESH

    def wrong_side_decision(self, active_lane_dir: Optional[Tuple[float, float]]) -> Optional[bool]:
        """
        Multi-frame voting for wrong-side detection.

        A single velocity vector is susceptible to noise; a sustained majority
        vote over WRONG_SIDE_VOTE_WINDOW frames is required before flagging.
        """
        if active_lane_dir is None:
            return None
        vx, vy = self.velocity_vector()
        speed  = (vx**2 + vy**2) ** 0.5
        if speed > PIXEL_MOVE_THRESH:
            dot = vx*active_lane_dir[0] + vy*active_lane_dir[1]
            self.direction_votes.append(dot < 0)
        votes = list(self.direction_votes)
        if len(votes) < max(2, WRONG_SIDE_VOTE_WINDOW // 2):
            return None
        return (sum(votes) / len(votes)) >= WRONG_SIDE_VOTE_THRESH

    def stationary_duration(self, fps: float, tolerance: float = 0.85) -> float:
        sample_rate = fps / FRAME_SAMPLE_RATE
        lookback = max(int(PARKING_STILL_SECS * sample_rate) + 5, 2)
        window = self.boxes[-lookback:]
        if len(window) < 2: return 0.0
        still_flags = [
            np.linalg.norm(
                np.array(self.center(window[i])) - np.array(self.center(window[i-1]))
            ) < PIXEL_MOVE_THRESH
            for i in range(1, len(window))
        ]
        return len(still_flags) / sample_rate if sum(still_flags)/len(still_flags) >= tolerance else 0.0


class Tracker:
    """
    Dual ByteTrack wrapper with separate tracker instances per vehicle class group.

    Motorcycles require permissive tracking settings due to low detection confidence
    and frequent partial occlusion. Large vehicles require strict settings to prevent
    road-paint markings from spawning ghost tracks. A single ByteTrack instance cannot
    satisfy both constraints simultaneously.

    Track IDs from each instance are kept disjoint by adding LV_TRACK_ID_OFFSET to all
    large-vehicle IDs, since each ByteTrack counter starts independently from 1.
    """

    TWO_WHEELER_PERSON_IDS = frozenset({0, 1, 3})   # person, bicycle, motorcycle
    LARGE_VEH_IDS          = frozenset({2, 5, 7})   # car, bus, truck

    def __init__(self, fps: float = 25.0):
        self.moto_tracker = sv.ByteTrack(
            track_activation_threshold = MOTO_TRACK_ACTIVATION_THRESH,
            lost_track_buffer          = MOTO_LOST_TRACK_BUFFER,
            minimum_matching_threshold = 0.80,
            frame_rate                 = int(fps),
            minimum_consecutive_frames = MOTO_MIN_CONSECUTIVE_FRAMES,
        )
        self.lv_tracker = sv.ByteTrack(
            track_activation_threshold = LV_TRACK_ACTIVATION_THRESH,
            lost_track_buffer          = LV_LOST_TRACK_BUFFER,
            minimum_matching_threshold = 0.80,
            frame_rate                 = int(fps),
            minimum_consecutive_frames = LV_MIN_CONSECUTIVE_FRAMES,
        )
        self.states: Dict[int, TrackState] = {}
        self.fps = fps

    def update(self, detections: sv.Detections,
               frame_idx: int, timestamp: float) -> Dict[int, TrackState]:

        if detections.class_id is None or len(detections) == 0:
            self._prune(frame_idx)
            return self.states

        moto_mask = np.isin(detections.class_id, list(self.TWO_WHEELER_PERSON_IDS))
        lv_mask   = np.isin(detections.class_id, list(self.LARGE_VEH_IDS))

        moto_dets = detections[moto_mask]
        lv_dets   = detections[lv_mask]

        groups = [
            (self.moto_tracker, moto_dets, 0),
            (self.lv_tracker,   lv_dets,   LV_TRACK_ID_OFFSET),
        ]
        for tracker_inst, dets, id_offset in groups:
            tracked = tracker_inst.update_with_detections(dets)
            if tracked.tracker_id is None or len(tracked) == 0:
                continue
            for i, raw_tid in enumerate(tracked.tracker_id):
                tid  = int(raw_tid) + id_offset
                cid  = int(tracked.class_id[i]) if tracked.class_id is not None else -1
                name = COCO_VEHICLE_IDS.get(cid, "person" if cid == COCO_PERSON_ID else "other")
                box  = tracked.xyxy[i]
                conf = float(tracked.confidence[i]) if tracked.confidence is not None else 0.0

                if tid not in self.states:
                    self.states[tid] = TrackState(track_id=tid, class_id=cid, class_name=name)
                s = self.states[tid]
                s.boxes.append(box)
                s.confidences.append(conf)
                s.frame_indices.append(frame_idx)
                s.timestamps.append(timestamp)

        self._prune(frame_idx)
        return self.states

    def _prune(self, frame_idx: int) -> None:
        """Remove TrackState entries that have not been updated within TRACK_MAX_AGE_FRAMES."""
        stale_ids = [
            tid for tid, s in self.states.items()
            if s.frame_indices and
               (frame_idx - s.frame_indices[-1]) > TRACK_MAX_AGE_FRAMES
        ]
        for tid in stale_ids:
            del self.states[tid]


print("TrackState + Tracker ready")


TrackState + Tracker ready


## 11. Violation Engine

`ViolationEngine.evaluate()` is called once per track per frame. It resolves the track's spatial position relative to the calibrated lane polygons and applies five independent rule checks. Each violation type is logged at most once per track to prevent duplicate events.

In [14]:
@dataclass
class ViolationEvent:
    track_id:       int
    class_name:     str
    violations:     List[str]
    frame_idx:      int
    timestamp:      float
    box:            np.ndarray
    confidence:     float
    plate_text:     str = ""
    evidence_frame: Optional[np.ndarray] = field(default=None, repr=False)


class ViolationEngine:
    """Rules engine: applies geometric and temporal violation criteria to each TrackState."""

    def __init__(self, calibration: SceneCalibration, fps: float = 25.0):
        self.calib = calibration
        self.fps   = fps
        self._flagged: Dict[int, set] = defaultdict(set)

    def evaluate(
        self,
        state:       TrackState,
        frame_idx:   int,
        timestamp:   float,
        light_color: str = "unknown",
    ) -> Optional[ViolationEvent]:

        box = state.latest_box()
        if box is None:
            return None

        x1, y1, x2, y2 = map(int, box)
        leading_edge = ((x1+x2)//2, y2)
        violations   = []

        in_lane_A = point_in_polygon(leading_edge, self.calib.lane_A)
        in_lane_B = point_in_polygon(leading_edge, self.calib.lane_B)
        active_lane_dir  = self.calib.lane_A_direction if in_lane_A else (
                           self.calib.lane_B_direction if in_lane_B else None)
        active_stop_line = self.calib.lane_A_stop_line if in_lane_A else (
                           self.calib.lane_B_stop_line if in_lane_B else None)

        # Rule 1: No helmet — evaluated via multi-frame vote to suppress single-frame misses
        if (ENABLE_HELMET
                and state.class_id in TWO_WHEELER_IDS
                and len(state.boxes) >= MIN_CONFIRM_FRAMES):
            decision = state.helmet_decision()
            if decision is False:
                if "no_helmet" not in self._flagged[state.track_id]:
                    violations.append("no_helmet")

        # Rule 2: Triple riding — flagged when rider count exceeds 2 in any of the last 5 frames
        if (ENABLE_TRIPLE_RIDING
                and state.class_id in TWO_WHEELER_IDS
                and state.rider_counts
                and max(state.rider_counts[-5:] or [0]) > 2):
            if "triple_riding" not in self._flagged[state.track_id]:
                violations.append("triple_riding")

        # Rule 3: Red-light violation — leading edge crosses stop-line polygon while light is red
        if ENABLE_STOP_LINE and light_color == "red" and active_stop_line is not None:
            if point_in_polygon(leading_edge, active_stop_line):
                if "red_light" not in self._flagged[state.track_id]:
                    violations.append("red_light")

        # Rule 4: Illegal parking — vehicle stationary in a no-parking zone for >= PARKING_STILL_SECS
        if (ENABLE_NO_PARKING
                and state.class_id != COCO_PERSON_ID
                and self.calib.no_parking_zones):
            still_secs = state.stationary_duration(self.fps)
            if still_secs >= PARKING_STILL_SECS:
                for zone in self.calib.no_parking_zones:
                    if point_in_polygon(leading_edge, zone):
                        if "illegal_parking" not in self._flagged[state.track_id]:
                            violations.append("illegal_parking")
                        break

        # Rule 5: Wrong-side driving — sustained directional vote against the assigned lane direction
        if ENABLE_WRONG_SIDE and state.class_id != COCO_PERSON_ID:
            decision = state.wrong_side_decision(active_lane_dir)
            if decision is True:
                if "wrong_side" not in self._flagged[state.track_id]:
                    violations.append("wrong_side")

        if not violations:
            return None

        self._flagged[state.track_id].update(violations)
        return ViolationEvent(
            track_id   = state.track_id,
            class_name = state.class_name,
            violations = violations,
            frame_idx  = frame_idx,
            timestamp  = timestamp,
            box        = box,
            confidence = state.confidences[-1] if state.confidences else 0.0,
        )


print("ViolationEngine ready")


ViolationEngine ready


## 12. License Plate OCR

`PlateOCR` performs text extraction on the lower portion of a vehicle bounding box. The crop is upscaled if its width falls below 120 px to improve character recognition accuracy. Extracted text is matched against an Indian registration plate pattern; unmatched strings are truncated and stored as-is.

In [15]:
import re

PLATE_PATTERN = re.compile(r"[A-Z]{2}\s?\d{1,2}\s?[A-Z]{1,2}\s?\d{4}")


class PlateOCR:
    """
    EasyOCR wrapper for license plate extraction.

    Initialisation is deferred to the first call to avoid loading the OCR model
    during setup. The reader is configured for CPU inference to preserve GPU
    memory for YOLO and helmet model inference.
    """

    def __init__(self):
        self._reader = None

    @property
    def reader(self):
        if self._reader is None:
            print("Initialising EasyOCR (first call only) ...")
            self._reader = easyocr.Reader(["en"], gpu=False)
            print("OCR ready")
        return self._reader

    def read_plate(self, crop: np.ndarray) -> str:
        if crop is None or crop.size == 0:
            return ""
        h, w = crop.shape[:2]
        if w < 120:
            crop = cv2.resize(crop, None,
                              fx=120/w, fy=120/w, interpolation=cv2.INTER_CUBIC)
        results = self.reader.readtext(crop, detail=0, paragraph=False)
        raw     = " ".join(results).upper().strip()
        match   = PLATE_PATTERN.search(raw)
        return match.group(0).replace(" ", "") if match else raw[:12]


plate_ocr = PlateOCR()
print("PlateOCR ready (reader loads on first violation)")


PlateOCR ready (reader loads on first violation)


## 13. Rendering and Evidence Storage

`draw_dashboard_overlay` renders lane polygons, no-parking zones, stop lines, and per-track bounding boxes onto every output frame. Ghost tracks are suppressed based on `TRACK_DISPLAY_AGE`.

`render_evidence` produces a single annotated frame for each violation event, combining the dashboard overlay with violation-specific annotations and plate/metadata text.

`EvidenceStore` writes annotated frames to disk and appends structured records to a JSONL log file, flushing after each write to ensure durability during long pipeline runs.

In [16]:
VIOLATION_COLORS = {
    "no_helmet":       (0,   0,   255),
    "triple_riding":   (0,   165, 255),
    "red_light":       (0,   0,   200),
    "illegal_parking": (255, 0,   255),
    "wrong_side":      (255, 255, 0),
}


def draw_dashboard_overlay(frame: np.ndarray,
                            states: Dict[int, "TrackState"],
                            calib: SceneCalibration,
                            current_frame_idx: int = 0) -> np.ndarray:
    """
    Render calibration regions and confirmed track bounding boxes onto every frame.

    Tracks that have not been updated within TRACK_DISPLAY_AGE frames are
    suppressed to prevent stale overlays from persisting after a vehicle exits frame.
    """
    out = frame.copy()
    cv2.polylines(out, [calib.lane_A], True, (0,255,0), 3)
    cv2.polylines(out, [calib.lane_B], True, (255,0,0), 3)
    if ENABLE_NO_PARKING:
        for zone in calib.no_parking_zones:
            cv2.polylines(out, [zone], True, (0,0,255), 3)
    if ENABLE_STOP_LINE:
        if calib.lane_A_stop_line is not None:
            cv2.polylines(out, [calib.lane_A_stop_line], True, (0,200,200), 3)
        if calib.lane_B_stop_line is not None:
            cv2.polylines(out, [calib.lane_B_stop_line], True, (0,200,200), 3)
    for tid, state in states.items():
        if len(state.boxes) < MIN_CONFIRM_FRAMES:
            continue
        if state.frame_indices and                 (current_frame_idx - state.frame_indices[-1]) > TRACK_DISPLAY_AGE:
            continue
        box = state.latest_box()
        if box is None:
            continue
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(out, (x1,y1), (x2,y2), (0,255,255), 2)
        cv2.putText(out, f"ID {tid} {state.class_name}", (x1, max(0,y1-8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,255), 2)
    return out


def render_evidence(frame: np.ndarray,
                    event: ViolationEvent,
                    calib: SceneCalibration) -> np.ndarray:
    out = frame.copy()
    overlay = out.copy()
    cv2.polylines(overlay, [calib.lane_A], True, (0,255,0), 4)
    cv2.polylines(overlay, [calib.lane_B], True, (255,0,0), 4)
    for zone in calib.no_parking_zones:
        cv2.fillPoly(overlay, [zone], (0,0,180))
    cv2.addWeighted(overlay, 0.25, out, 0.75, 0, out)

    x1, y1, x2, y2 = map(int, event.box)
    color = VIOLATION_COLORS.get(event.violations[0], (0,0,255))
    cv2.rectangle(out, (x1,y1), (x2,y2), color, 2)

    for j, line in enumerate([
        f"ID:{event.track_id}  {event.class_name}",
        " | ".join(event.violations),
        f"Plate: {event.plate_text or 'N/A'}",
        f"T={event.timestamp:.1f}s  conf={event.confidence:.2f}",
    ]):
        cv2.putText(out, line, (x1, y1 - 10 - j*18),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)

    cv2.putText(out, f"Frame {event.frame_idx}",
                (10, out.shape[0]-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200,200,200), 1)
    return out


class EvidenceStore:
    def __init__(self, output_dir: Path = OUTPUT_DIR):
        self.dir      = output_dir
        self.log_path = output_dir / "violations.jsonl"
        self._log     = open(self.log_path, "w")
        self.count    = 0

    def save(self, event: ViolationEvent, annotated_frame: np.ndarray) -> dict:
        self.count += 1
        fname = f"viol_{self.count:04d}_id{event.track_id}_f{event.frame_idx}.jpg"
        cv2.imwrite(str(self.dir / "frames" / fname), annotated_frame)
        record = {
            "id":         self.count,
            "track_id":   event.track_id,
            "class":      event.class_name,
            "violations": event.violations,
            "timestamp":  round(event.timestamp, 2),
            "frame_idx":  event.frame_idx,
            "plate":      event.plate_text,
            "confidence": round(event.confidence, 3),
            "evidence":   fname,
        }
        self._log.write(json.dumps(record, default=str) + "\n")
        self._log.flush()
        return record

    def close(self):
        self._log.close()


print("Evidence generation ready")


Evidence generation ready


## 14. Pipeline Execution

`run_pipeline` orchestrates the complete processing loop. For each sampled frame it calls detection, tracking, helmet inference (for two-wheelers), violation evaluation, and evidence storage in sequence. The dashboard overlay is composited onto every output frame; violation annotations are layered on top when an event fires.

In [17]:
def run_pipeline(source=VIDEO_SOURCE, max_frames: int = 500, show_preview: bool = False):
    ingestion = FrameIngestion(source)
    w, h = ingestion.resolution
    fps = ingestion.fps
    calib = make_demo_calibration(w, h)
    tracker = Tracker(fps=fps)
    engine = ViolationEngine(calib, fps=fps)
    store = EvidenceStore()
    video_path = OUTPUT_DIR / "annotated_output_fixed.mp4"
    writer = cv2.VideoWriter(str(video_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    records = []; processed = 0; t_start = time.time(); light_color = "unknown"

    for f_idx, ts, raw in ingestion.frames():
        if max_frames and processed >= max_frames: break
        processed += 1
        clean = preprocess_frame(raw, lite=LIVE_FEED_MODE)
        frame_to_output = clean.copy()

        detections = detector.detect(clean)

        if ENABLE_STOP_LINE and detections.class_id is not None:
            tl_mask = detections.class_id == COCO_TRAFFIC_LIGHT
            if tl_mask.any():
                light_color = detector.traffic_light_color(clean, detections.xyxy[tl_mask][0])

        if detections.class_id is not None:
            keep_ids = set(COCO_VEHICLE_IDS) | {COCO_PERSON_ID}
            vehicle_dets = detections[np.isin(detections.class_id, list(keep_ids))]
            states = tracker.update(vehicle_dets, f_idx, ts)
            frame_to_output = draw_dashboard_overlay(clean.copy(), states, calib,
                                                     current_frame_idx=f_idx)

            for tid, state in states.items():
                if state.class_id in TWO_WHEELER_IDS and ENABLE_HELMET:
                    crop = crop_rider_region(raw, state.latest_box())
                    if crop is not None:
                        hres = helmet_model(crop, conf=HELMET_CONF, verbose=False)[0]
                        state.helmet_seen.append(len(hres.boxes) > 0 and int(hres.boxes.cls[0]) in HELMET_PRESENT_CLASSES)

                event = engine.evaluate(state, f_idx, ts, light_color)
                if event:
                    event.plate_text = plate_ocr.read_plate(detector.crop_plate_region(clean, event.box))
                    annotated_event_frame = render_evidence(frame_to_output, event, calib)
                    records.append(store.save(event, annotated_event_frame))
                    frame_to_output = annotated_event_frame

        writer.write(frame_to_output)

    ingestion.release(); store.close(); writer.release()
    print(f"Done. Violations: {len(records)}")
    return records

records = run_pipeline(source=VIDEO_PATH, max_frames=2000)


Initialising EasyOCR (first call only) ...
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteOCR ready
Done. Violations: 2


## 15. Results

`load_violation_log` deserialises the JSONL event log written by `EvidenceStore`. `report` prints a per-type breakdown and the set of unique plates observed. Individual records are printed below.

In [18]:
from collections import Counter

def load_violation_log(log_path: Path = OUTPUT_DIR / "violations.jsonl"):
    rows = []
    with open(log_path) as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

def report(rows):
    if not rows:
        print("No violations recorded yet.")
        return
    all_viols = [v for r in rows for v in r["violations"]]
    counts    = Counter(all_viols)
    print("\n══ VIOLATION SUMMARY ════════════════════")
    for vtype, n in counts.most_common():
        print(f"  {vtype:<22} {n:>4}")
    print(f"  {'TOTAL':<22} {len(rows):>4}")
    plates = [r["plate"] for r in rows if r["plate"]]
    print(f"\nUnique plates detected: {len(set(plates))}")
    if plates:
        print(f"  e.g. {', '.join(sorted(set(plates))[:5])}")


if 'records' in locals() and records:
    print("\n══ VIOLATION RECORDS ════════════════════")
    for record in records:
        print(record)
elif 'records' in locals() and not records:
    print("No violations recorded.")
else:
    print("Records variable not found. Run the pipeline first.")



══ VIOLATION RECORDS ════════════════════
{'id': 1, 'track_id': 50019, 'class': 'bus', 'violations': ['illegal_parking'], 'timestamp': 21.76, 'frame_idx': 544, 'plate': 'HENTN NA ROI', 'confidence': 0.882, 'evidence': 'viol_0001_id50019_f544.jpg'}
{'id': 2, 'track_id': 50031, 'class': 'bus', 'violations': ['illegal_parking'], 'timestamp': 33.6, 'frame_idx': 840, 'plate': 'SCNI', 'confidence': 0.738, 'evidence': 'viol_0002_id50031_f840.jpg'}


## 16. Evaluation Utilities

`evaluate_performance` computes precision, recall, and F1 for a given violation type by matching predictions to ground-truth records on `(frame_idx, track_id)` keys. `benchmark_fps` measures detection throughput over a fixed number of sampled frames.

In [19]:
from sklearn.metrics import classification_report

def evaluate_performance(ground_truth: List[dict], predictions: List[dict],
                          violation_type: str = "no_helmet"):
    gt_keys   = {(r["frame_idx"], r["track_id"]) for r in ground_truth
                 if r["violation"] == violation_type}
    pred_keys = {(r["frame_idx"], r["track_id"]) for r in predictions
                 if violation_type in r["violations"]}
    all_keys  = gt_keys | pred_keys
    y_true    = [1 if k in gt_keys   else 0 for k in all_keys]
    y_pred    = [1 if k in pred_keys else 0 for k in all_keys]
    print(f"\n── {violation_type.upper()} ──")
    print(classification_report(y_true, y_pred,
                                 target_names=["clean", violation_type],
                                 zero_division=0))


def benchmark_fps(source, n_frames: int = 100):
    """Measure end-to-end pipeline throughput over N sampled frames."""
    ing   = FrameIngestion(source)
    count = 0
    t0    = time.time()
    for _, _, raw in ing.frames():
        clean = preprocess_frame(raw, lite=LIVE_FEED_MODE)
        _     = detector.detect(clean)
        count += 1
        if count >= n_frames:
            break
    ing.release()
    elapsed = time.time() - t0
    print(f"{count} frames in {elapsed:.2f}s → {count/elapsed:.1f} fps")


print("Evaluation utilities defined.")
print("benchmark_fps(VIDEO_PATH) — measure detection throughput.")
print("evaluate_performance(ground_truth, records) — compute precision/recall against labelled data.")


Evaluation utilities defined.
benchmark_fps(VIDEO_PATH) — measure detection throughput.
evaluate_performance(ground_truth, records) — compute precision/recall against labelled data.


## 17. REST API and Frontend Bridge

A FastAPI application exposes the pipeline over HTTP and tunnels the endpoint through ngrok for access from a locally-served HTML control panel.

**Endpoints:**

| Method | Path | Description |
|--------|------|-------------|
| GET | `/` | Health check |
| GET / POST | `/config` | Read or update pipeline configuration |
| GET | `/status` | Current pipeline state and frame count |
| POST | `/start` | Start the pipeline in a background thread |
| GET | `/frame` | Latest annotated frame as base64 JPEG |
| GET | `/violations` | Full violation log |
| GET | `/evidence/{filename}` | Retrieve a specific evidence frame |

The `ngrok-skip-browser-warning` response header is injected by middleware to prevent ngrok's interstitial warning page from blocking cross-origin API calls from the HTML frontend.

# EMERALD — Frontend Connection Guide

## Prerequisites

- The EMERALD notebook running in Google Colab (GPU runtime)
- `emerald_frontend.html` opened locally in Chrome or Edge
- A free [ngrok](https://ngrok.com) account with your authtoken

---

## Setup Steps

### 1. Configure the ngrok token in Colab

In the API cell near the bottom of the notebook, replace the placeholder with your ngrok authtoken:

```python
NGROK_AUTHTOKEN = "your_token_here"
```

### 2. Run the notebook

Go to **Runtime → Run all** (`Ctrl+F9`). When the API cell completes, it prints a public URL:

```
EMERALD API is LIVE
URL → https://abc123.ngrok.io
```

Copy this URL. It changes every time the Colab session restarts.

### 3. Open the control panel

Double-click `emerald_frontend.html` to open it in your browser.

### 4. Connect

In the **"1 · Connect to Colab"** field, paste the ngrok URL and click **Connect**. The status indicator in the top-right turns green when the connection is successful.

### 5. Set the video path

In the **"2 · Video path"** field, enter the Colab-side path to your video:

```
/content/drive/MyDrive/your_folder/video.mp4
```

### 6. Send config and start

Click **↑ Send Config to Colab**, then **▶ Start Pipeline**. The live annotated frame appears in the main panel within a few seconds.

---

## Troubleshooting

| Symptom | Resolution |
|---|---|
| "Could not connect" | Verify the ngrok URL is current and the API cell ran without errors |
| Connected but no frame | Click **Start Pipeline** — frames only stream while the pipeline is active |
| URL stopped working | The Colab session disconnected; rerun the API cell and copy the new URL |
| No violations appearing | Click **↻ Refresh Violations** in the log panel |

---

> Evidence frames are stored in Colab at `violation_output/frames/`. Copy this folder to Google Drive before ending your session to retain the output.

In [20]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "fastapi", "uvicorn[standard]", "pyngrok==6.0.0",
                "python-multipart", "--quiet"], check=True)

import threading, base64, time, json
from pathlib import Path
from typing import Optional

import cv2, numpy as np
import uvicorn
from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok, conf as ngrok_conf

# Replace with a valid ngrok authtoken before running
NGROK_AUTHTOKEN = "3FR2zlUjR5QhkZl5gFtlKOKIbZH_7CPJg31LSnSAXB1okdccW"

# ── Shared state ──────────────────────────────────────────────────────────────
PIPELINE_CONFIG = {
    "enable_helmet":        True,
    "enable_wrong_side":    True,
    "enable_no_parking":    True,
    "enable_stop_line":     True,
    "enable_triple_riding": True,
    "lane_a":               [],
    "lane_b":               [],
    "parking_zones":        [],
    "lane_a_stop_line":     [],
    "lane_b_stop_line":     [],
    "video_path":           VIDEO_PATH,
    "max_frames":           2000,
}

PIPELINE_STATUS = {
    "running":          False,
    "processed_frames": 0,
    "violations_count": 0,
    "current_fps":      0.0,
    "light_color":      "unknown",
    "last_violation":   None,
}

_latest_frame_bytes: Optional[bytes] = None
_frame_lock = threading.Lock()

def set_latest_frame(frame: np.ndarray):
    global _latest_frame_bytes
    _, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 75])
    with _frame_lock:
        _latest_frame_bytes = buf.tobytes()

# ── FastAPI application ───────────────────────────────────────────────────────
app = FastAPI(title="EMERALD API", version="2.0")

# CORS is open to allow requests from a locally-served HTML frontend
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
    expose_headers=["*"],
)

# Attach ngrok-skip-browser-warning to all responses to bypass the ngrok interstitial page
@app.middleware("http")
async def add_ngrok_header(request: Request, call_next):
    response = await call_next(request)
    response.headers["ngrok-skip-browser-warning"] = "true"
    response.headers["Access-Control-Allow-Origin"] = "*"
    return response

class ConfigPayload(BaseModel):
    enable_helmet:        bool = True
    enable_wrong_side:    bool = True
    enable_no_parking:    bool = True
    enable_stop_line:     bool = True
    enable_triple_riding: bool = True
    lane_a:               list = []
    lane_b:               list = []
    parking_zones:        list = []
    lane_a_stop_line:     list = []
    lane_b_stop_line:     list = []
    video_path:           str  = ""
    max_frames:           int  = 2000

@app.get("/")
def root():
    return {"status": "EMERALD API running ✓", "version": "2.0"}

@app.post("/config")
def update_config(payload: ConfigPayload):
    global ENABLE_HELMET, ENABLE_WRONG_SIDE, ENABLE_NO_PARKING
    global ENABLE_STOP_LINE, ENABLE_TRIPLE_RIDING
    PIPELINE_CONFIG.update(payload.dict())
    ENABLE_HELMET        = payload.enable_helmet
    ENABLE_WRONG_SIDE    = payload.enable_wrong_side
    ENABLE_NO_PARKING    = payload.enable_no_parking
    ENABLE_STOP_LINE     = payload.enable_stop_line
    ENABLE_TRIPLE_RIDING = payload.enable_triple_riding
    if payload.video_path:
        PIPELINE_CONFIG["video_path"] = payload.video_path
    return {"ok": True}

@app.get("/config")
def get_config():
    return PIPELINE_CONFIG

@app.get("/status")
def get_status():
    return PIPELINE_STATUS

@app.get("/frame")
def get_frame():
    with _frame_lock:
        data = _latest_frame_bytes
    if data is None:
        raise HTTPException(status_code=404, detail="No frame yet")
    return {"frame_b64": base64.b64encode(data).decode("utf-8")}

@app.get("/violations")
def get_violations():
    log_path = OUTPUT_DIR / "violations.jsonl"
    if not log_path.exists():
        return {"violations": [], "total": 0}
    rows = []
    with open(log_path) as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return {"violations": rows, "total": len(rows)}

@app.get("/evidence/{filename}")
def get_evidence_frame(filename: str):
    fpath = OUTPUT_DIR / "frames" / filename
    if not fpath.exists():
        raise HTTPException(status_code=404, detail="Frame not found")
    encoded = base64.b64encode(fpath.read_bytes()).decode("utf-8")
    return {"frame_b64": encoded, "filename": filename}

@app.post("/start")
def start_pipeline():
    if PIPELINE_STATUS["running"]:
        return {"ok": False, "reason": "Pipeline already running"}

    cfg = PIPELINE_CONFIG

    def _arr(pts):
        return np.array(pts, dtype=np.int32) if pts else None

    def _run():
        PIPELINE_STATUS["running"] = True
        try:
            src = cfg["video_path"] or VIDEO_PATH
            cap_tmp = cv2.VideoCapture(src)
            fw = int(cap_tmp.get(cv2.CAP_PROP_FRAME_WIDTH))
            fh = int(cap_tmp.get(cv2.CAP_PROP_FRAME_HEIGHT))
            cap_tmp.release()

            calib_override = make_demo_calibration(fw, fh)
            if cfg["lane_a"]:        calib_override.lane_A              = _arr(cfg["lane_a"])
            if cfg["lane_b"]:        calib_override.lane_B              = _arr(cfg["lane_b"])
            if cfg["parking_zones"]: calib_override.no_parking_zones    = [_arr(z) for z in cfg["parking_zones"]]
            if cfg["lane_a_stop_line"]: calib_override.lane_A_stop_line = _arr(cfg["lane_a_stop_line"])
            if cfg["lane_b_stop_line"]: calib_override.lane_B_stop_line = _arr(cfg["lane_b_stop_line"])

            run_pipeline_api(source=src, max_frames=cfg["max_frames"],
                             calib_override=calib_override)
        finally:
            PIPELINE_STATUS["running"] = False

    threading.Thread(target=_run, daemon=True).start()
    return {"ok": True, "message": "Pipeline started"}

# ── API-aware pipeline wrapper ────────────────────────────────────────────────
# Mirrors run_pipeline but writes frame bytes to _latest_frame_bytes for
# the /frame endpoint and updates PIPELINE_STATUS every 10 processed frames.
def run_pipeline_api(source, max_frames, calib_override=None):
    ingestion = FrameIngestion(source)
    w, h = ingestion.resolution
    fps  = ingestion.fps
    calib   = calib_override or make_demo_calibration(w, h)
    tracker = Tracker(fps=fps)
    engine  = ViolationEngine(calib, fps=fps)
    store   = EvidenceStore()

    writer = cv2.VideoWriter(
        str(OUTPUT_DIR / "annotated_output_api.mp4"),
        cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h)
    )

    processed = 0; t_start = time.time(); light_color = "unknown"

    try:
        for f_idx, ts, raw in ingestion.frames():
            if max_frames and processed >= max_frames:
                break
            processed += 1
            clean     = preprocess_frame(raw, lite=LIVE_FEED_MODE)
            frame_out = clean.copy()
            detections = detector.detect(clean)

            if ENABLE_STOP_LINE and detections.class_id is not None:
                tl_mask = detections.class_id == COCO_TRAFFIC_LIGHT
                if tl_mask.any():
                    light_color = detector.traffic_light_color(clean, detections.xyxy[tl_mask][0])

            if detections.class_id is not None:
                keep_ids = set(COCO_VEHICLE_IDS) | {COCO_PERSON_ID}
                vehicle_dets = detections[np.isin(detections.class_id, list(keep_ids))]
                states = tracker.update(vehicle_dets, f_idx, ts)
                frame_out = draw_dashboard_overlay(clean.copy(), states, calib)

                for tid, state in states.items():
                    if state.class_id in TWO_WHEELER_IDS and ENABLE_HELMET:
                        crop = crop_rider_region(raw, state.latest_box())
                        if crop is not None:
                            hres = helmet_model(crop, conf=HELMET_CONF, verbose=False)[0]
                            state.helmet_seen.append(
                                len(hres.boxes) > 0
                                and int(hres.boxes.cls[0]) in HELMET_PRESENT_CLASSES
                            )
                    event = engine.evaluate(state, f_idx, ts, light_color)
                    if event:
                        event.plate_text = plate_ocr.read_plate(
                            detector.crop_plate_region(clean, event.box))
                        annotated = render_evidence(frame_out, event, calib)
                        rec       = store.save(event, annotated)
                        frame_out = annotated
                        PIPELINE_STATUS["violations_count"] += 1
                        PIPELINE_STATUS["last_violation"]    = rec

            writer.write(frame_out)
            set_latest_frame(frame_out)

            if processed % 10 == 0:
                elapsed = time.time() - t_start
                PIPELINE_STATUS["processed_frames"] = processed
                PIPELINE_STATUS["current_fps"]      = round(processed / max(elapsed, 1), 1)
                PIPELINE_STATUS["light_color"]      = light_color

    finally:
        ingestion.release(); store.close(); writer.release()
        PIPELINE_STATUS["running"] = False
    print(f"Pipeline done — {processed} frames, {PIPELINE_STATUS['violations_count']} violations.")

# ── Server startup ────────────────────────────────────────────────────────────
ngrok.set_auth_token(NGROK_AUTHTOKEN)
try:
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
except Exception:
    pass

# bind_tls=True enforces HTTPS, preventing mixed-content errors when the
# frontend is served over a secure origin and calls the API tunnel.
tunnel = ngrok.connect(
    addr=8000,
    bind_tls=True,
)
PUBLIC_URL = tunnel.public_url

print("\n" + "═" * 58)
print("  EMERALD API is LIVE")
print(f"  URL  →  {PUBLIC_URL}")
print("═" * 58)
print("  1. Copy the URL above.")
print("  2. Paste it into the frontend Colab URL field.")
print("  3. Connect → Send Config → Start Pipeline.")
print("═" * 58 + "\n")

def _serve():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=_serve, daemon=True).start()
time.sleep(2)
print("Server ready.")



══════════════════════════════════════════════════════════
  EMERALD API is LIVE
  URL  →  https://electable-emphasis-radiantly.ngrok-free.dev
══════════════════════════════════════════════════════════
  1. Copy the URL above.
  2. Paste it into the frontend Colab URL field.
  3. Connect → Send Config → Start Pipeline.
══════════════════════════════════════════════════════════

Server ready.
